<a href="https://colab.research.google.com/github/vitoriaferreirap/DeepLearning/blob/main/CNN_Computer_Vision/03_DCL_Treinamento_de_Anotacoes_Frames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Objetivo desta etapa:
- Treinar a arquitetura ResNet-50 para reconhecer pontos anatômicos (cascos/articulações) nos frames selecionados. O resultado esperado é uma rede capaz de automatizar a anotação dos frames restantes do vídeo. Analisar a aplicação de hiperparâmetros diferentes, análise de métricas e diversidade de banco de dados.
- Treinamentos serão feitos com dados locais, para uma melhor performance e maior velocidade, em seguida copiados no Drive.
- Criação de gráfico Loss para verificar a performance de treino.
- Análise, métricas de treino e validação, estarão no arquivo PerformanceIndicators.md do repositório: https://github.com/vitoriaferreirap/DeepLearning/blob/main/PerformanceIndicators.md


In [ ]:
!unzip -q /content/EstimativaPose-Cavalos-2026-03-06.zip -d /content/EstimativaPose_LOCAL

In [ ]:
# traz para os arquivos para pasta principal
!mv /content/EstimativaPose_LOCAL/EstimativaPose-Cavalos-2026-03-06/* /content/EstimativaPose_LOCAL/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# O ponto (.) indica "copie o conteúdo para aqui, na raiz atual"
!cp -r /content/drive/MyDrive/deeplabcut/EstimativaPose_BACKUP/. /content/EstimativaPose_BACKUP

In [ ]:
!pip install --pre deeplabcut

In [ ]:
import deeplabcut

## Create a training dataset:


In [ ]:
path_config_local = 'EstimativaPose_LOCAL/config.yaml'

deeplabcut.create_training_dataset(
    path_config_local,
    net_type="resnet_50",
    num_shuffles=1,
    engine=deeplabcut.Engine.PYTORCH,
    augmenter_type='imgaug'
)

In [ ]:
project_folder_local = "EstimativaPose_BACKUP"

videofile_path = [
  f"/content/{project_folder_local}/videos/1.mp4",
  f"/content/{project_folder_local}/videos/2.mp4",
  f"/content/{project_folder_local}/videos/3.mp4",
  f"/content/{project_folder_local}/videos/4.mp4",
  f"/content/{project_folder_local}/videos/5.mp4",
  f"/content/{project_folder_local}/videos/6.mp4",
  f"/content/{project_folder_local}/videos/7.mp4",
  f"/content/{project_folder_local}/videos/8.mp4",
  f"/content/{project_folder_local}/videos/9.mp4",
  f"/content/{project_folder_local}/videos/10.mp4"
]

# Onde os vídeos analisados serão salvos temporariamente
destfolder = f"/content/{project_folder_local}/labeled-videos"

# O arquivo de configuração que o DeepLabCut vai ler para o treino
path_config_file = f"/content/{project_folder_local}/config.yaml"

print(f"Caminho do Config Local: {path_config_file}")
print(f"Primeiro vídeo local: {videofile_path[0]}")

## Treinamento

In [ ]:
deeplabcut.train_network(
    path_config_file,
    shuffle=1,
    trainingsetindex=0,
    device="cuda:0",
    max_snapshots_to_keep=5,
    displayiters=100,
    save_epochs=100,
    epochs=700,
    saveiters=500
)

## Análise


In [ ]:
deeplabcut.evaluate_network(path_config_file, plotting=True)

## Gráfico de Confiança (Likelihood)

In [ ]:
import os
from IPython.display import Image, display

# Busca automática e exibição direta
encontrado = False
for root, dirs, files in os.walk(f"/content/EstimativaPose_BACKUP"):
    for file in files:
        if "likelihood" in file.lower() and file.endswith(".png"):
            caminho = os.path.join(root, file)
            print(f"Exibindo diagnóstico: {caminho}")
            display(Image(filename=caminho))
            encontrado = True
            break # Mostra apenas o primeiro para não encher a tela
    if encontrado: break

if not encontrado:
    print("Gráfico não encontrado. Rode: deeplabcut.plot_trajectories(path_config_file, videofile_path, destfolder=destfolder)")

## Cria gráfico Loss


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

csv_path = "/content/EstimativaPose_BACKUP/dlc-models-pytorch/iteration-0/EstimativaPoseMar6-trainset95shuffle1/train/learning_stats.csv"

if os.path.exists(csv_path):
    # Lendo o CSV e garantindo que os dados sejam numéricos
    try:
        data = pd.read_csv(csv_path, header=None)

        # O DLC salva 3 colunas: Iteração, Loss, LR
        # Vamos converter para numérico, transformando erros em 'NaN' e depois removendo-os
        data[0] = pd.to_numeric(data[0], errors='coerce')
        data[1] = pd.to_numeric(data[1], errors='coerce')
        data = data.dropna()

        plt.figure(figsize=(12, 6))
        plt.plot(data[0], data[1], color='teal', linewidth=2, label='Treinamento (Loss)')

        plt.title('Curva de Aprendizado - Estimativa de Pose (Cavalos)', fontsize=14)
        plt.xlabel('Iterações', fontsize=12)
        plt.ylabel('Loss', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.6)

        # Escala logarítmica é melhor para ver o Loss chegando perto de zero
        plt.yscale('log')
        plt.legend()

        plt.show()
        print(f"Gráfico gerado! Total de pontos processados: {len(data)}")

    except Exception as e:
        print(f"Erro ao processar os dados: {e}")
else:
    print("Arquivo learning_stats.csv não encontrado no caminho.")

## Análise vídeos:
Esta função analisa o novo vídeo. O usuário pode escolher o melhor modelo a partir dos resultados da avaliação e especificar o índice correto do snapshot para a variável ` snapshotindex` no arquivo `config.yaml` . Caso contrário, por padrão, o snapshot mais recente é usado para analisar o vídeo.

Os resultados são armazenados em um arquivo hd5 no mesmo diretório onde o vídeo está localizado.

In [ ]:
deeplabcut.analyze_videos(
    path_config_file,
    videofile_path,
    videotype='.mp4',
    destfolder=destfolder,
)

## Trace as trajetórias dos vídeos analisados:
Esta função traça as trajetórias de todas as partes do corpo ao longo de todo o vídeo. Cada parte do corpo é identificada por uma cor única.

In [ ]:
deeplabcut.plot_trajectories(
    path_config_file,
    videofile_path,
    videotype='.mp4',
    destfolder=destfolder,
)

## Criar vídeo com legendas:
Esta função serve para fins de visualização e pode ser usada para criar um vídeo em formato .mp4 com rótulos previstos pela rede. Este vídeo é salvo no mesmo diretório onde o vídeo original está localizado.

In [ ]:
deeplabcut.create_labeled_video(
    path_config_file,
    videofile_path,
    videotype='.mp4',
    destfolder=destfolder,
)